## Notebook 06 - Political ideology labeling (Left / Center / Right) + manual audit

**Goal.** MIND does not provide publisher/outlet names reliably, so we cannot directly map each article to an outlet-ideology label.  
To keep the project’s political-bias study meaningful, I **infer an ideology label** from the **article text (title + abstract)** using a **pretrained classifier**, and then run a **small manual audit (30–50 items)** to validate that this labeling step is not random.

**Main decision produced here**
- Downstream notebooks will use **`center` vs `non_center`** as the primary ideology variable (`ideology_used`).
- 3-class labels (`left/center/right`) are kept for optional exploratory analysis and reproducibility.

**Artifacts produced**
- `data/processed/political_ideology_train.pkl` and `data/processed/political_ideology_dev.pkl`
- `data/processed/audit_sample.csv` and `data/processed/audit_results.csv`
- `data/processed/political_ideology_*_3class_highconf_0.55.pkl`



### Imports and Paths

In [25]:
from pathlib import Path
import pandas as pd
import numpy as np
import sklearn
from sklearn.metrics import confusion_matrix, classification_report, cohen_kappa_score, accuracy_score, f1_score

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

PROJECT = Path.cwd().parents[0]  
DATA_DIR = PROJECT / "data"
PROCESSED = DATA_DIR / "processed"

print("PROJECT:", PROJECT)
print("PROCESSED:", PROCESSED, "exists?", PROCESSED.exists())

PROCESSED.mkdir(parents=True, exist_ok=True)

print("torch:", torch.__version__)
print("transformers:", __import__("transformers").__version__)
print("sklearn:", sklearn.__version__)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)


PROJECT: c:\Users\jlsmp\Documents\universidade\M.IA\IAS\project
PROCESSED: c:\Users\jlsmp\Documents\universidade\M.IA\IAS\project\data\processed exists? True
torch: 2.9.1+cpu
transformers: 4.57.3
sklearn: 1.8.0
DEVICE: cpu


### Loading

In [4]:
news_train = pd.read_pickle(PROCESSED / "news_train.pkl")
news_dev   = pd.read_pickle(PROCESSED / "news_dev.pkl")

print("news_train:", news_train.shape)
print("news_dev:  ", news_dev.shape)

# Optional
clicks_train_path = PROCESSED / "clicks_train.pkl"
clicks_dev_path   = PROCESSED / "clicks_dev.pkl"

clicks_train = pd.read_pickle(clicks_train_path) if clicks_train_path.exists() else None
clicks_dev   = pd.read_pickle(clicks_dev_path) if clicks_dev_path.exists() else None

print("clicks_train:", None if clicks_train is None else clicks_train.shape)
print("clicks_dev:  ", None if clicks_dev is None else clicks_dev.shape)


news_train: (51282, 8)
news_dev:   (42416, 8)
clicks_train: (5843444, 9)
clicks_dev:   (2740998, 9)


### Define politics

In [6]:
def add_politics_flag(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["is_politics"] = out["subcategory"].astype(str).str.contains("polit", case=False, na=False)
    return out

news_train2 = add_politics_flag(news_train)
news_dev2   = add_politics_flag(news_dev)

print("Political items (train):", int(news_train2["is_politics"].sum()))
print("Political items (dev):  ", int(news_dev2["is_politics"].sum()))

pol_train = news_train2.loc[news_train2["is_politics"]].copy()
pol_dev   = news_dev2.loc[news_dev2["is_politics"]].copy()

pol_train[["news_id","category","subcategory","title","abstract"]].head()


Political items (train): 2831
Political items (dev):   2402


,news_id,category,subcategory,title,abstract
20,N9786,news,newspolitics,Elijah Cummings to lie in state at US Capitol ...,"Cummings, a Democrat whose district included s..."
127,N47214,news,newspolitics,Here are the lawmakers who are not seeking ree...,The battle for control of Congress is more tha...
160,N24905,news,newspolitics,Grieder: Special election in House District 28...,The special election in Texas House District 2...
182,N56618,news,newspolitics,CNN and MSNBC hit Ingraham guest for accusing ...,Anchors on CNN and MSNBC expressed their outra...
260,N34406,news,newspolitics,Sen. Sinema's new bill would allow families' n...,"Currently, only veterans are entitled to heads..."


### Load a pretrained Left/Center/Right classifier

I used `bucketresearch/politicalBiasBERT` (Hugging Face), which outputs:
- 0 → left  
- 1 → center  
- 2 → right  

> This is not “ground truth”. It’s a proxy label that will be validated with a small manual audit.

In [7]:
MODEL_NAME = "bucketresearch/politicalBiasBERT"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

id2label = {0:"left", 1:"center", 2:"right"}


config.json:   0%|          | 0.00/909 [00:00<?, ?B/s]

c:\Users\jlsmp\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jlsmp\.cache\huggingface\hub\models--bucketresearch--politicalBiasBERT. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer.json: 0.00B [00:00, ?B/s]

c:\Users\jlsmp\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

### Run ideology inference (batching)

Each article is classified using **title + abstract**.

Why batching?
- Much faster than one-by-one
- Uses vectorization on CPU/GPU

I also store the model confidence (max softmax probability), so later I can:
- Inspect low-confidence cases
- Optionally filter very uncertain predictions

In [8]:
def build_text(df: pd.DataFrame) -> pd.Series:
    title = df["title"].fillna("").astype(str)
    abstract = df["abstract"].fillna("").astype(str)
    # Use a separator so the model sees two fields
    return (title + " [SEP] " + abstract).str.strip()

@torch.no_grad()
def predict_ideology(texts, batch_size=32):
    preds = []
    confs = []
    probs_all = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = tokenizer(
            batch,
            truncation=True,
            padding=True,
            max_length=256,
            return_tensors="pt"
        ).to(DEVICE)

        logits = model(**enc).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()

        y = probs.argmax(axis=1)
        c = probs.max(axis=1)

        preds.extend(y.tolist())
        confs.extend(c.tolist())
        probs_all.append(probs)

    probs_all = np.vstack(probs_all)
    return np.array(preds), np.array(confs), probs_all

# Train politics
texts_train = build_text(pol_train).tolist()
pred_train, conf_train, probs_train = predict_ideology(texts_train, batch_size=32)

pol_train["ideology_pred"] = [id2label[i] for i in pred_train]
pol_train["ideology_conf"] = conf_train
pol_train["p_left"]   = probs_train[:,0]
pol_train["p_center"] = probs_train[:,1]
pol_train["p_right"]  = probs_train[:,2]

pol_train[["news_id","ideology_pred","ideology_conf","title"]].head()


,news_id,ideology_pred,ideology_conf,title
20,N9786,right,0.480308,Elijah Cummings to lie in state at US Capitol ...
127,N47214,left,0.565310,Here are the lawmakers who are not seeking ree...
160,N24905,left,0.852884,Grieder: Special election in House District 28...
182,N56618,right,0.704687,CNN and MSNBC hit Ingraham guest for accusing ...
260,N34406,right,0.517555,Sen. Sinema's new bill would allow families' n...


In [9]:
# Dev politics
texts_dev = build_text(pol_dev).tolist()
pred_dev, conf_dev, probs_dev = predict_ideology(texts_dev, batch_size=32)

pol_dev["ideology_pred"] = [id2label[i] for i in pred_dev]
pol_dev["ideology_conf"] = conf_dev
pol_dev["p_left"]   = probs_dev[:,0]
pol_dev["p_center"] = probs_dev[:,1]
pol_dev["p_right"]  = probs_dev[:,2]

pol_dev[["news_id","ideology_pred","ideology_conf","title"]].head()


,news_id,ideology_pred,ideology_conf,title
22,N9786,right,0.480308,Elijah Cummings to lie in state at US Capitol ...
73,N21802,left,0.448908,The Democratic candidates who want to face Trump
83,N18066,right,0.502502,Viral anti-Trump moment costs woman her job
140,N47214,left,0.565310,Here are the lawmakers who are not seeking ree...
177,N24905,left,0.852884,Grieder: Special election in House District 28...


###  Save ideology labels (so other notebooks can reuse them)

I save only the political subset with ideology predictions.

In [10]:
cols_to_save = ["news_id","ideology_pred","ideology_conf","p_left","p_center","p_right","title","abstract","category","subcategory"]

pol_train_out = pol_train[cols_to_save].copy()
pol_dev_out   = pol_dev[cols_to_save].copy()

pol_train_out.to_pickle(PROCESSED / "political_ideology_train.pkl")
pol_dev_out.to_pickle(PROCESSED / "political_ideology_dev.pkl")

print("Saved:")
print("-", PROCESSED / "political_ideology_train.pkl")
print("-", PROCESSED / "political_ideology_dev.pkl")


Saved:
- c:\Users\jlsmp\Documents\universidade\M.IA\IAS\project\data\processed\political_ideology_train.pkl
- c:\Users\jlsmp\Documents\universidade\M.IA\IAS\project\data\processed\political_ideology_dev.pkl


### Quick sanity checks (distribution + examples)

- Are predictions extremely skewed to one label? (bad sign)
- Are confidences reasonable?
- Do example titles look plausible?

In [11]:
def summarize_preds(df, name="train"):
    print(f"== {name} ==")
    print("Counts:")
    print(df["ideology_pred"].value_counts(dropna=False))
    print("\nMean confidence:", df["ideology_conf"].mean())
    print("Median confidence:", df["ideology_conf"].median())

summarize_preds(pol_train_out, "train politics")
summarize_preds(pol_dev_out, "dev politics")

for lab in ["left","center","right"]:
    ex = pol_train_out.loc[pol_train_out["ideology_pred"]==lab].sort_values("ideology_conf", ascending=False).head(3)
    print("\n---", lab.upper(), "(high confidence examples) ---")
    display(ex[["ideology_conf","title"]])


== train politics ==
Counts:
ideology_pred
right     1571
left       922
center     338
Name: count, dtype: int64

Mean confidence: 0.5763864238137535
Median confidence: 0.5437219738960266
== dev politics ==
Counts:
ideology_pred
right     1384
left       752
center     266
Name: count, dtype: int64

Mean confidence: 0.5744563157760928
Median confidence: 0.5410856008529663

--- LEFT (high confidence examples) ---


,ideology_conf,title
13431,0.997102,New Florida Majority celebrates grand opening ...
36993,0.996141,Opinion | 'Read the transcript': How Republica...
37401,0.996060,Rude Kellyanne Conway trashes gracious Wolf Bl...



--- CENTER (high confidence examples) ---


,ideology_conf,title
10018,0.988912,Impeachment inquiry approval reaches highest l...
33956,0.988195,Biden will always represent the 'safety candid...
16897,0.987718,Sen. Jon Cornyn says impeachment push indicate...



--- RIGHT (high confidence examples) ---


,ideology_conf,title
3124,0.886640,Defense Secretary: U.S. Tried To 'Dissuade' Tu...
37055,0.882931,Top White House official told Congress 'there ...
5150,0.882311,House ethics committee investigating allegatio...


The predicted ideology distribution in the politics subset is stable across train/dev, suggesting consistent behavior rather than randomness.

However, the average confidence is modest (~0.54–0.58), indicating that 3-way ideology labeling is inherently noisy for many items.

High-confidence examples look plausible, but confidence behaves differently by class (left/center can be extremely high, right tends to have a lower ceiling), so confidence filtering may change the class balance.

This supports later decision: use center vs non_center as the primary ideology variable, and use 3-class only as exploratory or high-confidence subset analysis

### Manual audit: create a balanced labeling sample (15/15/15)

To estimate how reliable the pretrained ideology classifier is, I manually label a small sample of political news and compare human labels against model predictions.

This cell creates an audit file with:
- **45 items total**, balanced across the model's predicted classes
  (`15 left`, `15 center`, `15 right`) to ensure each class is represented.
- A shuffled order (to reduce labeling bias).
- Empty columns `human_label` and `notes` for manual annotation.

The output is written to `data/processed/audit_sample.csv`. 

In [44]:
RNG = np.random.default_rng(42)
N_TOTAL = 45  

parts = []
for lab, n in [("left", 15), ("center", 15), ("right", 15)]:
    subset = pol_train_out.loc[pol_train_out["ideology_pred"] == lab].copy()
    if len(subset) == 0:
        continue
    take = min(n, len(subset))
    idx = RNG.choice(subset.index.to_numpy(), size=take, replace=False)
    parts.append(subset.loc[idx])

audit = pd.concat(parts, ignore_index=True)
audit = audit.sample(frac=1.0, random_state=42).reset_index(drop=True)  # shuffle

audit["human_label"] = ""  
audit["notes"] = ""

audit_path = PROCESSED / "audit_sample.csv"
if audit_path.exists():
    print("Audit file already exists, not overwriting:", audit_path)
else:
    audit.to_csv(audit_path, index=False)
    print("Wrote:", audit_path)
audit.head(10)


Audit file already exists, not overwriting: c:\Users\jlsmp\Documents\universidade\M.IA\IAS\project\data\processed\audit_sample.csv


,news_id,ideology_pred,ideology_conf,p_left,p_center,p_right,title,abstract,category,subcategory,human_label,notes
0,N43515,right,0.490145,0.298340,0.211515,0.490145,Jimmy Carter In Hospital To Relieve Brain Pres...,Former President Jimmy Carter is in a Georgia ...,news,newspolitics,,
1,N36886,center,0.449383,0.281425,0.449383,0.269192,Statue of George H.W. Bush's beloved service d...,A life-size statue of George H.W. Bush's belov...,news,newspolitics,,
2,N19095,center,0.988195,0.003753,0.988195,0.008053,Biden will always represent the 'safety candid...,Democratic strategist Estuardo Rodriguez said ...,news,newspolitics,,
3,N59965,right,0.727887,0.197551,0.074562,0.727887,White House to add two aides to lead impeachme...,The White House is expected to add a pair of a...,news,newspolitics,,
4,N48114,right,0.734055,0.166846,0.099099,0.734055,Staffer overheard Trump ask Sondland about 'in...,"William Taylor, the top U.S. diplomat in Ukrai...",news,newspolitics,,
5,N25812,right,0.438231,0.161498,0.400271,0.438231,L.A.-Area High School Students Walk Out in Sup...,A Los Angeles-based advocacy group is leading ...,news,newspolitics,,
6,N4417,left,0.816606,0.816606,0.084082,0.099312,FiveThirtyEight: Thoughts on GOP Convention Day 1,"In her hometown of Cleveland, Clare Malone ref...",news,newspolitics,,
7,N21920,left,0.481211,0.481211,0.135898,0.382891,How potential new White House hopefuls could r...,Former Massachusetts Gov. Deval Patrick and Ne...,news,newspolitics,,
8,N35386,left,0.637044,0.637044,0.160353,0.202604,The Trump chaos theory for how to beat impeach...,Analysis: While Democrats need a clear message...,news,newspolitics,,
9,N53377,left,0.558501,0.558501,0.248756,0.192743,Deval Patrick in race to make Democrat debate ...,Newly-announced 2020 candidate Deval Patrick n...,news,newspolitics,,


### Next step
Open `audit_sample.csv` and fill `human_label` with one of:
- `left`, `center`, `right`

I created `notes` to record ambiguity or reasoning but didn't really used it.
Then I run the next cell to compute agreement metrics.

### Manual audit: evaluate model predictions vs human labels

After filling `human_label` in `audit_sample.csv`, I measure agreement between:
- `human_label` (manual ground truth)
- `ideology_pred` (model prediction)

Steps:
1) Load the filled audit file
2) Normalize label formatting (strip + lowercase)
3) Keep only valid labels (`left`, `center`, `right`)
4) Compute:
   - **Accuracy**
   - **Cohen’s κ (kappa)** for agreement beyond chance
   - Confusion matrix + per-class precision/recall/F1

We also save the cleaned labeled subset to `audit_results.csv` for later analysis.


In [15]:
filled_path = PROCESSED / "audit_sample.csv" 
aud = pd.read_csv(filled_path)

aud["human_label"] = aud["human_label"].astype(str).str.strip().str.lower()

valid = aud[aud["human_label"].isin(["left","center","right"])].copy()

print("Audit rows total:", len(aud))
print("Usable (not unsure):", len(valid))

if len(valid) == 0:
    print("Fill human_label first, then rerun.")
else:
    y_true = valid["human_label"]
    y_pred = valid["ideology_pred"]

    print("\nAccuracy:", (y_true == y_pred).mean())
    print("Cohen kappa:", cohen_kappa_score(y_true, y_pred, labels=["left","center","right"]))

    print("\nConfusion matrix (rows=true, cols=pred):")
    cm = confusion_matrix(y_true, y_pred, labels=["left","center","right"])
    cm_df = pd.DataFrame(cm, index=["true_left","true_center","true_right"], columns=["pred_left","pred_center","pred_right"])
    display(cm_df)

    print("\nClassification report:")
    print(classification_report(y_true, y_pred, labels=["left","center","right"]))

    out_path = PROCESSED / "audit_results.csv"
    valid.to_csv(out_path, index=False)
    print("\nSaved cleaned audit rows:", out_path)


Audit rows total: 45
Usable (not unsure): 45

Accuracy: 0.6222222222222222
Cohen kappa: 0.43333333333333335

Confusion matrix (rows=true, cols=pred):


,pred_left,pred_center,pred_right
true_left,10,3,6
true_center,2,11,2
true_right,3,1,7



Classification report:
              precision    recall  f1-score   support

        left       0.67      0.53      0.59        19
      center       0.73      0.73      0.73        15
       right       0.47      0.64      0.54        11

    accuracy                           0.62        45
   macro avg       0.62      0.63      0.62        45
weighted avg       0.64      0.62      0.62        45


Saved cleaned audit rows: c:\Users\jlsmp\Documents\universidade\M.IA\IAS\project\data\processed\audit_results.csv


- Accuracy = 0.62 and Cohen’s κ = 0.43 (moderate agreement beyond chance).
- Performance is strongest for **center** (F1=0.73), suggesting the model can reasonably detect neutral reporting.
- The main failure mode is **confusion between left and right** (e.g., 6/19 true-left items predicted as right), meaning ideology labels are noisy at the extremes.

Conclusion: the classifier provides **weak ideology labels** suitable for exploratory analysis, but results must be interpreted cautiously. To reduce noise, later analyses should use confidence filtering (high-confidence labels) and/or consider grouping strategies (e.g., center vs non-center).

### Qualitative observations from manual labeling (human audit)

Based on the manual annotation of the audit sample, a clear pattern emerged:

#### Center (neutral) articles are easier to identify
Neutral/center articles were often straightforward to label because they tended to:
- use **more descriptive, factual language** (e.g., reporting what happened rather than evaluating it),
- avoid strongly opinionated wording (for example, fewer **loaded adjectives**),
- refer to **both sides** in a balanced way (e.g., mentioning *Republicans* and *Democrats* in the same context), or
- avoid partisan references altogether when the content was purely informational.

This aligns with the quantitative audit results, where the model achieved its strongest performance on the `center` class.

#### Left vs right is harder and error-prone
When articles referenced vocabulary associated with **both political sides**, distinguishing `left` from `right` was more difficult.
A common ambiguous case was:
- the text mentions both sides, but the **tone** (e.g., negative framing, blame, sarcasm, or selective emphasis)
  is directed mainly at one side.

This kind of distinction relies heavily on **pragmatic cues** and **subtle sentiment/framing**, which can be challenging
for automated models and may explain part of the confusion observed between `left` and `right` in the confusion matrix.

#### Implication for downstream analysis
Given that `center` is comparatively easier and more stable, while `left` vs `right` contains more ambiguity, the project will:
- prioritize a **binary ideology signal** (`center` vs `non_center`) for the main bias/reinforcement analysis, and
- treat `left/center/right` primarily as an **exploratory** view, optionally restricted to a high-confidence subset.


### Audit evaluation: 3-class vs binary ideology reliability

- **Binary ideology**: `center` vs `non_center` (where `non_center = left ∪ right`)

Now I want to see if this labeling is more accurate and reliable.

This helps decide whether downstream bias analyses should rely on the full `left/center/right` labels or a more robust `center/non_center` formulation.


In [23]:
def add_binary_center_noncenter(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    # non-center probability: sum of extremes
    out["p_non_center"] = out["p_left"] + out["p_right"]

    # binary prediction
    out["ideology_bin_pred"] = np.where(
        out["p_center"] >= out["p_non_center"],
        "center",
        "non_center"
    )

    # binary confidence + margin (useful for filtering)
    out["ideology_bin_conf"] = out[["p_center", "p_non_center"]].max(axis=1)
    out["ideology_bin_margin"] = (out[["p_center", "p_non_center"]].max(axis=1)
                                 - out[["p_center", "p_non_center"]].min(axis=1))
    return out

pol_train = add_binary_center_noncenter(pol_train)
pol_dev   = add_binary_center_noncenter(pol_dev)

pol_train[["ideology_pred","ideology_conf","ideology_bin_pred","ideology_bin_conf"]].head()


,ideology_pred,ideology_conf,ideology_bin_pred,ideology_bin_conf
20,right,0.480308,non_center,0.821202
127,left,0.565310,non_center,0.816298
160,left,0.852884,non_center,0.903028
182,right,0.704687,non_center,0.877811
260,right,0.517555,non_center,0.802376


The table illustrates how the same news item can have **low confidence** in the full
`left/center/right` prediction while still having **high confidence** in the simplified
binary label `center` vs `non_center`.

- `ideology_pred` / `ideology_conf` refer to the **3-class** classifier output.
  Here, values around ~0.48–0.56 mean the model is **not strongly certain** about whether the
  item is specifically *left* or *right* (and it may be competing with *center*).

- `ideology_bin_pred` / `ideology_bin_conf` collapse `left ∪ right` into **non_center**.
  In these examples, the binary confidence is much higher (~0.80–0.90), meaning the model is
  **consistently confident that the content is not center**, even if it is unsure about the
  exact direction (left vs right).

This supports the decision to use **center vs non_center** as the primary ideology label for
downstream bias analysis, because it is more stable under uncertainty than the full 3-class
labeling.

### Audit evaluation: 3-class ideology vs center/non-center (binary) + confidence filtering


Finally, I test **confidence filtering** to study the trade-off between:
- **Coverage** (how many audited items we keep) and
- **Quality** (how accurate the remaining predictions are).

Two filtering strategies are evaluated:
- **Confidence threshold** (`*_conf`): keep items whose top predicted probability ≥ threshold.
- **Margin threshold** (`*_margin`): keep items whose top probability minus second-best probability ≥ threshold.

The filtering table reports, for each threshold:
- `kept_n` / `kept_frac`: how many audited items remain after filtering.
- performance metrics on that filtered subset (accuracy and κ/macro-F1 for 3-class, F1 for binary).


In [26]:
cols_to_save = [
    "news_id",
    "ideology_pred","ideology_conf","p_left","p_center","p_right",
    "p_non_center","ideology_bin_pred","ideology_bin_conf","ideology_bin_margin",
    "title","abstract","category","subcategory"
]


In [34]:
audit_path = PROCESSED / "audit_results.csv"   
audit = pd.read_csv(audit_path)

def detect_prob_cols(df: pd.DataFrame):
    candidates = [
        ["p_left", "p_center", "p_right"],
        ["prob_left", "prob_center", "prob_right"],
    ]
    for cols in candidates:
        if all(c in df.columns for c in cols):
            return cols
    raise ValueError(f"Could not find probability columns. Columns are: {list(df.columns)}")

pcols = detect_prob_cols(audit)
pL, pC, pR = pcols

# --- 3-class confidence + margin ---
probs = audit[[pL, pC, pR]].to_numpy()
sorted_probs = np.sort(probs, axis=1)
audit["conf_3"] = sorted_probs[:, -1]
audit["margin_3"] = sorted_probs[:, -1] - sorted_probs[:, -2]

# --- binary center vs non-center from probs ---
audit["p_non_center"] = audit[pL] + audit[pR]
audit["bin_pred_prob"] = np.where(audit[pC] >= audit["p_non_center"], "center", "non_center")
audit["conf_bin"] = audit[[pC, "p_non_center"]].max(axis=1)
audit["margin_bin"] = (audit[[pC, "p_non_center"]].max(axis=1)
                       - audit[[pC, "p_non_center"]].min(axis=1))

audit["bin_true"] = np.where(audit["human_label"] == "center", "center", "non_center")
audit["bin_pred_from_label"] = np.where(audit["ideology_pred"] == "center", "center", "non_center")

print("\n=== Binary (center vs non-center) ===")
print("Accuracy:", accuracy_score(audit["bin_true"], audit["bin_pred_from_label"]))
print("F1 (non_center as positive):", f1_score(
    audit["bin_true"].map({"center":0,"non_center":1}),
    audit["bin_pred_from_label"].map({"center":0,"non_center":1})
))

print("\nConfusion matrix (binary) rows=true, cols=pred:")
cm = confusion_matrix(audit["bin_true"], audit["bin_pred_from_label"], labels=["center","non_center"])
display(pd.DataFrame(cm, index=["true_center","true_non_center"], columns=["pred_center","pred_non_center"]))

def eval_filtered(df, conf_col, margin_col, conf_thr=None, margin_thr=None, y_true_col=None, y_pred_col=None, labels=None):
    keep = np.ones(len(df), dtype=bool)
    if conf_thr is not None:
        keep &= df[conf_col].to_numpy() >= conf_thr
    if margin_thr is not None:
        keep &= df[margin_col].to_numpy() >= margin_thr

    sub = df[keep].copy()
    if len(sub) == 0:
        return None

    y_true = sub[y_true_col]
    y_pred = sub[y_pred_col]

    res = {
        "kept_n": len(sub),
        "kept_frac": len(sub)/len(df),
        "accuracy": accuracy_score(y_true, y_pred),
    }
    if labels is not None and len(labels) > 2:
        res["kappa"] = cohen_kappa_score(y_true, y_pred, labels=labels)
        res["macro_f1"] = f1_score(y_true, y_pred, labels=labels, average="macro")
    else:
        yt = y_true.map({"center":0,"non_center":1})
        yp = y_pred.map({"center":0,"non_center":1})
        res["f1_non_center"] = f1_score(yt, yp)
    return res

rows = []

for conf_thr in [0.50, 0.55, 0.60]:
    rows.append(("3class_conf", conf_thr, eval_filtered(
        audit, "conf_3", "margin_3", conf_thr=conf_thr,
        y_true_col="human_label", y_pred_col="ideology_pred",
        labels=["left","center","right"]
    )))
for margin_thr in [0.10, 0.15, 0.20]:
    rows.append(("3class_margin", margin_thr, eval_filtered(
        audit, "conf_3", "margin_3", margin_thr=margin_thr,
        y_true_col="human_label", y_pred_col="ideology_pred",
        labels=["left","center","right"]
    )))

for conf_thr in [0.55, 0.60, 0.65]:
    rows.append(("bin_conf", conf_thr, eval_filtered(
        audit, "conf_bin", "margin_bin", conf_thr=conf_thr,
        y_true_col="bin_true", y_pred_col="bin_pred_from_label",
        labels=["center","non_center"]
    )))
for margin_thr in [0.10, 0.15, 0.20]:
    rows.append(("bin_margin", margin_thr, eval_filtered(
        audit, "conf_bin", "margin_bin", margin_thr=margin_thr,
        y_true_col="bin_true", y_pred_col="bin_pred_from_label",
        labels=["center","non_center"]
    )))

out = []
for kind, thr, res in rows:
    if res is None:
        continue
    out.append({"filter": kind, "threshold": thr, **res})

display(pd.DataFrame(out).sort_values(["filter","threshold"]))



=== Binary (center vs non-center) ===
Accuracy: 0.8222222222222222
F1 (non_center as positive): 0.8666666666666667

Confusion matrix (binary) rows=true, cols=pred:


,pred_center,pred_non_center
true_center,11,4
true_non_center,4,26


,filter,threshold,kept_n,kept_frac,accuracy,kappa,macro_f1,f1_non_center
0,3class_conf,0.50,31,0.688889,0.741935,0.612500,0.742424,NaN
1,3class_conf,0.55,26,0.577778,0.807692,0.712389,0.810185,NaN
2,3class_conf,0.60,21,0.466667,0.809524,0.712329,0.819697,NaN
3,3class_margin,0.10,37,0.822222,0.675676,0.514754,0.672642,NaN
4,3class_margin,0.15,34,0.755556,0.705882,0.559585,0.704875,NaN
5,3class_margin,0.20,28,0.622222,0.750000,0.624521,0.753457,NaN
6,bin_conf,0.55,42,0.933333,0.857143,NaN,NaN,0.896552
7,bin_conf,0.60,35,0.777778,0.857143,NaN,NaN,0.909091
8,bin_conf,0.65,34,0.755556,0.852941,NaN,NaN,0.909091
9,bin_margin,0.10,42,0.933333,0.857143,NaN,NaN,0.896552


### Binary (center vs non-center)
When collapsing ideology into **center vs non_center**, performance improves substantially:
**Accuracy ≈ 0.82** and **F1(non_center) ≈ 0.87**.  
The confusion matrix shows relatively few mistakes in both directions (some centers flagged as
non-center and some non-centers missed), but overall the binary label is much more stable than the 3-class label.


### Confidence filtering trade-off
The filtering results show a clear pattern:

- For **3-class**, requiring higher confidence (e.g., `ideology_conf ≥ 0.55`) improves reliability
  (Accuracy ≈ 0.81 and κ ≈ 0.71), but reduces coverage (only ~58% of audited items kept).
  → 3-class can be high-quality **only on a smaller subset**.

- For **binary**, high reliability is achieved **without losing much coverage**
  (e.g., `ideology_bin_conf ≥ 0.55` keeps ~93% while Accuracy ≈ 0.86).
  → Binary labels remain strong **at scale**.

### Final choice for downstream analysis
Given the observed reliability, the project will use:
- **Ideology label:** `center` vs `non_center` (robust and high coverage)
- maybe a secondary one with: `ideology_pred` (left/center/right), on a **high-confidence subset**

### Export: ideology-labeled datasets for downstream notebooks

I export the ideology predictions in a clean, reusable format so the next notebooks can load a single file and use the selected ideology variable (`center` vs `non_center`) consistently.

Artifacts saved:
- `political_ideology_train.pkl` / `political_ideology_dev.pkl` (political subset + ideology columns)



In [35]:
FINAL_COLS = [
    "news_id",
    "title", "abstract", "category", "subcategory",
    "ideology_pred", "ideology_conf", "p_left", "p_center", "p_right",
    "p_non_center", "ideology_bin_pred", "ideology_bin_conf", "ideology_bin_margin",
]

required = ["news_id","ideology_pred","ideology_conf","p_left","p_center","p_right","ideology_bin_pred","ideology_bin_conf","p_non_center"]
missing = [c for c in required if c not in pol_train.columns]
assert not missing, f"Missing required columns in pol_train: {missing}"

pol_train_final = pol_train[FINAL_COLS].copy()
pol_dev_final   = pol_dev[FINAL_COLS].copy()

pol_train_final["ideology_used"] = pol_train_final["ideology_bin_pred"]
pol_dev_final["ideology_used"]   = pol_dev_final["ideology_bin_pred"]

pol_train_final.to_pickle(PROCESSED / "political_ideology_train.pkl")
pol_dev_final.to_pickle(PROCESSED / "political_ideology_dev.pkl")

pol_train_final.to_csv(PROCESSED / "political_ideology_train.csv", index=False)
pol_dev_final.to_csv(PROCESSED / "political_ideology_dev.csv", index=False)

print("Saved:")
print("-", PROCESSED / "political_ideology_train.pkl")
print("-", PROCESSED / "political_ideology_dev.pkl")
print("-", PROCESSED / "political_ideology_train.csv")
print("-", PROCESSED / "political_ideology_dev.csv")

print("\nTrain ideology_used distribution:")
display(pol_train_final["ideology_used"].value_counts(dropna=False).to_frame("count"))

print("\nDev ideology_used distribution:")
display(pol_dev_final["ideology_used"].value_counts(dropna=False).to_frame("count"))


Saved:
- c:\Users\jlsmp\Documents\universidade\M.IA\IAS\project\data\processed\political_ideology_train.pkl
- c:\Users\jlsmp\Documents\universidade\M.IA\IAS\project\data\processed\political_ideology_dev.pkl
- c:\Users\jlsmp\Documents\universidade\M.IA\IAS\project\data\processed\political_ideology_train.csv
- c:\Users\jlsmp\Documents\universidade\M.IA\IAS\project\data\processed\political_ideology_dev.csv

Train ideology_used distribution:


,count
ideology_used,
non_center,2631
center,200



Dev ideology_used distribution:


,count
ideology_used,
non_center,2246
center,156


### Sanity check: is the center/non-center imbalance caused by low confidence?

The ideology labels in the political subset are highly imbalanced (most items are labeled `non_center`).
To verify that this is not an artifact of low-confidence predictions, I examine:

1) The distribution of `ideology_bin_conf` separately for items predicted as `center` vs `non_center`.
2) How the class proportions change when we filter to **high-confidence** predictions
   (e.g., keep only items with `ideology_bin_conf ≥ 0.55` or `≥ 0.70`).

If the imbalance were caused by uncertain predictions, we would expect confidence filtering to
remove many `non_center` items and increase the relative share of `center`.


In [37]:
train = pd.read_pickle(PROCESSED / "political_ideology_train.pkl")

summary = train.groupby("ideology_used")["ideology_bin_conf"].describe()
display(summary)

# How many center items do we get if we require higher confidence?
for thr in [0.55, 0.60, 0.65, 0.70]:
    kept = train[train["ideology_bin_conf"] >= thr]
    counts = kept["ideology_used"].value_counts(normalize=False)
    frac = kept["ideology_used"].value_counts(normalize=True)
    print(f"\nThreshold >= {thr}: kept {len(kept)}/{len(train)}")
    display(pd.DataFrame({"count": counts, "share": frac}))

,count,mean,std,min,25%,50%,75%,max
ideology_used,,,,,,,,
center,200.0,0.754427,0.154574,0.504084,0.602778,0.761876,0.896964,0.988912
non_center,2631.0,0.803866,0.098249,0.502337,0.751448,0.821393,0.873386,0.998567



Threshold >= 0.55: kept 2762/2831


,count,share
ideology_used,,
non_center,2583,0.935192
center,179,0.064808



Threshold >= 0.6: kept 2652/2831


,count,share
ideology_used,,
non_center,2499,0.942308
center,153,0.057692



Threshold >= 0.65: kept 2534/2831


,count,share
ideology_used,,
non_center,2400,0.947119
center,134,0.052881



Threshold >= 0.7: kept 2366/2831


,count,share
ideology_used,,
non_center,2245,0.948859
center,121,0.051141


### Interpretation: the imbalance is not driven by uncertainty

Both `center` and `non_center` predictions have relatively high average confidence, and `non_center`
is (on average) even more confident than `center`.

Moreover, applying stricter confidence thresholds (e.g., `ideology_bin_conf ≥ 0.70`) does **not**
increase the proportion of `center`, instead, it slightly decreases it. This indicates that the
observed skew toward `non_center` is not explained by low-confidence noise, but rather reflects how
the model (and/or the dataset’s political content) is being classified.

Implication for downstream analysis:
we should focus on **relative changes** (e.g., recommendation/click shares compared to catalog
baseline) rather than interpreting the raw proportion of `non_center` as a standalone result.


### Optional: high-confidence 3-class subset

Although the main analysis uses `center` vs `non_center`, I also create a **high-confidence** subset for `left/center/right` to enable limited exploratory comparisons with reduced label noise.

Then check:
- dataset size after filtering,
- class distribution (balance),
- how filtering shifts the class proportions relative to the full set.


In [42]:
HC_THR = 0.55
train_3class_hc = pol_train_final[pol_train_final["ideology_conf"] >= HC_THR].copy()
dev_3class_hc   = pol_dev_final[pol_dev_final["ideology_conf"] >= HC_THR].copy()

train_3class_hc.to_pickle(PROCESSED / f"political_ideology_train_3class_highconf_{HC_THR}.pkl")
dev_3class_hc.to_pickle(PROCESSED / f"political_ideology_dev_3class_highconf_{HC_THR}.pkl")

print("Saved high-confidence 3-class subsets (threshold =", HC_THR, ")")


Saved high-confidence 3-class subsets (threshold = 0.55 )


In [43]:
def show_balance_change(full_df, hc_df, col="ideology_pred", title=""):
    a = full_df[col].value_counts(normalize=True).rename("full_share")
    b = hc_df[col].value_counts(normalize=True).rename("hc_share")
    out = pd.concat([a, b], axis=1).fillna(0) * 100
    out["delta_pp"] = out["hc_share"] - out["full_share"]
    print(title)
    display(out.round(2))

show_balance_change(pol_train_final, train_3class_hc, title="Train: share change after high-confidence filtering")
show_balance_change(pol_dev_final, dev_3class_hc, title="Dev: share change after high-confidence filtering")

for name, df in [("train_3class_hc", train_3class_hc), ("dev_3class_hc", dev_3class_hc)]:
    print("\n===", name, "===")
    print("Rows:", len(df))

    print("3-class distribution (counts):")
    display(df["ideology_pred"].value_counts(dropna=False).to_frame("count"))

    print("3-class distribution (share):")
    display((df["ideology_pred"].value_counts(normalize=True) * 100).round(2).to_frame("percent"))

    print("Confidence by predicted class:")
    display(df.groupby("ideology_pred")["ideology_conf"].describe())


Train: share change after high-confidence filtering


,full_share,hc_share,delta_pp
ideology_pred,,,
right,55.49,59.3,3.80
left,32.57,27.6,-4.97
center,11.94,13.1,1.16


Dev: share change after high-confidence filtering


,full_share,hc_share,delta_pp
ideology_pred,,,
right,57.62,63.19,5.57
left,31.31,25.07,-6.24
center,11.07,11.74,0.67



=== train_3class_hc ===
Rows: 1366
3-class distribution (counts):


...

,count
ideology_pred,
right,721
left,286
center,134


3-class distribution (share):


,percent
ideology_pred,
right,63.19
left,25.07
center,11.74


Confidence by predicted class:


,count,mean,std,min,25%,50%,75%,max
ideology_pred,,,,,,,,
center,134.0,0.780760,0.142163,0.554890,0.647587,0.792287,0.912719,0.987552
left,286.0,0.720201,0.140152,0.550794,0.599023,0.675085,0.843803,0.995460
right,721.0,0.671282,0.083944,0.550072,0.597444,0.660772,0.734481,0.888984


### Interpretation: high-confidence 3-class subset is usable (but shifts class proportions)

Filtering to `ideology_conf ≥ 0.55` yields sizable subsets (≈1.3k train, ≈1.1k dev), with all three classes present in meaningful counts. This makes the subset suitable for exploratory `left/center/right` analyses.

However, confidence filtering changes the class distribution: the share of `right` increases while `left` decreases, indicating that the model is more likely to exceed the confidence threshold on items predicted as `right`. Therefore, any 3-class results should be interpreted as applying to a **high-confidence subset** rather than the full political set.


## Conclusion

In this notebook, I generated ideology labels for the political subset of the dataset using a
pretrained `left/center/right` classifier applied to each article’s title and abstract. I then
validated the reliability of these labels through a **manual audit** of 45 sampled items.

The audit shows **moderate agreement** for the full 3-class task (accuracy ≈ 0.62; Cohen’s κ ≈ 0.43), with the strongest performance on the `center` class and more confusion between `left` and `right`.
When collapsing ideology into a binary formulation (**center vs non_center**), reliability improves
substantially (accuracy ≈ 0.82; F1(non_center) ≈ 0.87), making this binary signal the most robust
choice for downstream bias analysis.

I also studied confidence filtering and found that a **high-confidence 3-class subset**
(`ideology_conf ≥ 0.55`) is large enough to support exploratory analysis, but it changes the class
distribution (e.g., increasing the share of items predicted as `right`). For this reason, 3-class
results are treated as secondary and should be interpreted only within the filtered subset.

Finally, I exported standardized ideology-labeled datasets (`political_ideology_train/dev.*`) with a
single downstream label column (`ideology_used`) set to the chosen **center vs non_center**
operationalization.


## Next notebook: ideology-aware bias and reinforcement analysis

The next notebook uses the exported ideology-labeled datasets to quantify recommendation bias and
potential reinforcement dynamics.

Using `ideology_used` (center vs non_center), I will compute and compare ideology distributions at
different stages of the pipeline:

- **Catalog baseline:** ideology composition of the available political news.
- **Exposure / recommendations:** ideology composition of what the system shows in top-k.
- **Clicks / engagement:** ideology composition of what users choose to consume.

Because the political subset is highly skewed toward `non_center`, the key quantity is not the raw
share alone but **amplification relative to the baseline**, e.g.:

- Δ Exposure = P(non_center | recommended) − P(non_center | catalog)
- Δ Clicks   = P(non_center | clicked) − P(non_center | recommended)

I will also examine these quantities across user groups (e.g., politically engaged vs not) and over
time (when temporal logs are available), to test whether interactions can shift exposure toward more
non-center content.
